***Runtime Notes***

For this notebook, running on the CPU will work but is if model inference is slow, consider using the T4 GPU.

***How to use***
1. Run all cells
2. Upload or import image data
3. Use the is_deep_fake(ing_path) function to detect deep fake images.
  - Returns True: Image is deep fake
  - Returns False: Image is real

# Setup

## Imports

In [ ]:
from typing import List

import numpy as np

from PIL import Image
import cv2

from torchvision.transforms import v2 as T
import torchvision.transforms.functional as F

import torch
import torch.nn as nn
import torchvision.models as models

## Load Model

### Get Model File

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Path to model (Each group member may need to modify this)
MODEL_IN_DRIVE = "/content/drive/MyDrive/COMP-SCI 5530 - Principles of Data Science/Project/training/best_model/model_state_dict.pth"

In [ ]:
# Copy model from Google Drive
!cp "{MODEL_IN_DRIVE}" "/content/model_state_dict.pth"

### Transforms

In [ ]:
class ResizeAndPad:
    def __init__(self, target_size, fill=0):
        self.target_h, self.target_w = target_size
        self.fill = fill

    def __call__(self, img):
        w, h = img.size  # PIL: (width, height)

        if w > self.target_w:
          img.thumbnail((self.target_w, h))
          w, h = img.size  # PIL: (width, height)
        if h > self.target_h:
          img.thumbnail((w, self.target_h))
          w, h = img.size  # PIL: (width, height)

        pad_h = max(self.target_h - h, 0)
        pad_w = max(self.target_w - w, 0)

        # Split padding evenly (center the image)
        padding = (
            pad_w // 2,                    # left
            pad_h // 2,                    # top
            pad_w - pad_w // 2,            # right
            pad_h - pad_h // 2             # bottom
        )

        return F.pad(img, padding, fill=self.fill)

In [ ]:
img_size = 150

# Base transforms for val, test, and inference
base_transforms = T.Compose([
  ResizeAndPad((img_size, img_size)),

  T.ToImage(),
  T.ToDtype(torch.float32, scale=True),

  T.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
  )
])

### Model

In [ ]:
class DeepFakeDetector(nn.Module):
  def __init__(self):
    super().__init__()

    backbone = models.resnet50(pretrained=False)

    self.features = nn.Sequential(*list(backbone.children())[:-1])

    self.classifier = nn.Sequential(
      nn.Linear(2048, 512),
      nn.ReLU(),
      nn.Dropout(0.2),
      nn.Linear(512, 1)
    )

  def forward(self, x):
    x = self.features(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x

## Setup GPU

In [ ]:
# Set device to GPU if available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

## Load model

In [ ]:
# Set model
model = DeepFakeDetector()

# Load model weights
model.load_state_dict(torch.load('model_state_dict.pth', map_location=device))

# Move model to device
model = model.to(device)

# Set model to inference mode
model.eval()

# Inference Functions

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def get_faces(img_path: str) -> List[np.ndarray]:
    image = cv2.imread(img_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces_bboxes = face_cascade.detectMultiScale(gray, 1.3, 5)

    face_images = []
    for (x, y, w, h) in faces_bboxes:
        cropped_face = image[y:y+h, x:x+w]
        face_images.append(cropped_face)

    return face_images

**Model Output Classes**

0: Deep fake image

1: Real imag

***Function Returns***

True: Deep fake image detected

False: Real image detected

In [ ]:
# Example inference
def is_deep_fake_image(img_path: str) -> bool:
  image = Image.open(img_path).convert("RGB") # Load image

  # Get faces from image
  faces = get_faces(img_path)

  pred_classes = [] # List of predicted classes form faces

  for face in faces:
    image = Image.fromarray(face) # Convert to PIL image
    image = base_transforms(image) # Apply transforms
    image = image.unsqueeze(0) # Add batch dimension
    output = model(image) # Get prediction from model, output is raw logit
    if(output.numel() != 0): # Make sure we actually got an output from the model
      prob = torch.sigmoid(output) # Get probability converting logit to sigmoid
      print(f"Probability of real iamge: {prob.item():.2f}")
      pred_class = (prob > 0.5).long().item() # Convert probability sigmoid to class 0=deep_fake 1=real
      pred_classes.append(pred_class)

  if pred_classes:
    # Convert class to string and report resutl
    if 0 in pred_classes: # Image contains deep fake face
      print("Deep fake image detected")
      return True
    else: # No deep fake faces found
      print("Real image detected")
      return False

  print("Unable to determine is image is real or deep fake")
  return True # Unable to tell if real or not so return that it is a deep fake

# inference Example

You may need to comment out the print lines in the above function is you run a lot of images through it.

In [ ]:
is_deep_fake_image("/content/fake_8.jpg")